intended to analyze output from run_METRICS_TESTING.sh

***to-do:***
- [ ] change analysis to be easily digestable by others (formatted output showing specific answers to questions and clearly state what its showing)





***possible future tasks if I need something to do:***
- look into posibility of detecting other games running based on resorce usage? (this way, can run tests at same time as others if necessary)

- add ability to dynamically split both on batch and label, allowing analysis of both within-batch comparisons and between-batch comparisons
    - perhaps include auto-detection for above? (lower priority)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import os

import glob
import json

sys.path.insert(0, os.path.abspath('.'))
#/home/theandyman/IPD/IPD/work/forge/llm/IPD-LLM-Agents2/forgedb.py
from forgedb import ForgeDB


In [ ]:
# Check datapath for files
db = ForgeDB()

#=================================================================================================================
# for below paths, game folders are named with the same pattern as the metrics files, 
# so we can match them up when loading data
#=================================================================================================================
testing_games_path = './outputs/testing/games/' # contains folders with game data from that batch
testing_metrics_path = './outputs/testing/metrics/' # contains files with metrics data from that batch

expirementing_games_path = './outputs/experimenting/games/' # contains folders with game data from that batch
expirementing_metrics_path = './outputs/experimenting/metrics/' # contains files with metrics



## reference from Doug's analysis code
# data_path = './results/dhart/results/'

# print(f"Path: {data_path}")
# print(f"Exists: {os.path.exists(data_path)}")
# print(f"Is dir: {os.path.isdir(data_path)}")
# print(f"Files: {os.listdir(data_path) if os.path.isdir(data_path) else 'N/A'}")

# #results = db.load_batch(data_path)
# results = db.load_batch(data_path, pattern='*ep50_r10_t0.2*')
# print(f"Loaded: {len(results['loaded'])}")
# print(f"Skipped: {len(results['skipped'])}")
# print(f"Failed: {len(results['failed'])}")

# # Get episode data from database
# df = db.get_episode_summary(username='dhart', filename='%ep50_r10_t0.2%')
# df2 = db.get_summary(username='dhart', filename='%ep50_r10_t0.2%')
# print('Experiment Summary')
# display(df2)

# print(f"Found {df['results_id'].nunique()} sessions")
# db.close()

---
## Data Loading

Four file types are loaded for each mode (`testing`, `experimenting`), producing **5 DataFrames per mode** (10 total):

| DataFrame | Source pattern | Granularity |
|---|---|---|
| `{mode}_metrics_csv` | `*_game_metrics.csv` | one row per game run |
| `{mode}_metrics_gpu` | `*_continuous_gpu.csv` | one row per GPU per ~2s sample |
| `{mode}_games_summary` | `episodic_game_*.json` | one row per game file |
| `{mode}_games_episodes` | `episodic_game_*.json` | one row per episode per game |
| `{mode}_games_rounds` | `episodic_game_*.json` | one row per round per episode per game |

`games_log` paths are also indexed in `{mode}_games_log` for error-checking purposes (data is redundant with JSON).

**Join key across all DataFrames:** `session_id` (`YYYYMMDD_HHMMSS` timestamp string)

In [ ]:
# ── Loader functions and GPU column names ──────────────────────────────────────
# ── GPU column names ───────────────────────────────────────────────────────────
# Source file uses two comment lines (#) as its header — not parseable by pandas
# directly. We skip them with comment='#' and supply names manually.
# Units: fb/bar1/ccpm in MB; sm/mem/enc/dec in %; jpg/ofa not available on all hardware.
GPU_COLUMNS = ['Date', 'Time', 'gpu_idx', 'fb_MB', 'bar1_MB', 'ccpm_MB',
               'sm_pct', 'mem_pct', 'enc_pct', 'dec_pct', 'jpg', 'ofa']


# ── Loaders ────────────────────────────────────────────────────────────────────

def load_metrics_csv(metrics_path):
    """One row per game run.
    Renames batch_id → session_id so all DataFrames share the same join key.
    """
    dfs = []
    for f in glob.glob(os.path.join(metrics_path, '*_game_metrics.csv')):
        dfs.append(pd.read_csv(f))
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df = df.rename(columns={'batch_id': 'session_id'})
    return df


def load_metrics_gpu(metrics_path):
    """Continuous GPU telemetry. One row per GPU index per ~2s sample.
    session_id and server are parsed from filename:
        {YYYYMMDD}_{HHMMSS}_{server}_continuous_gpu.csv
    """
    dfs = []
    for f in glob.glob(os.path.join(metrics_path, '*_continuous_gpu.csv')):
        parts = os.path.basename(f).split('_')
        session_id = f"{parts[0]}_{parts[1]}"
        server     = parts[2]

        df = pd.read_csv(
            f,
            comment='#',
            header=None,
            names=GPU_COLUMNS,
            skipinitialspace=True,
            na_values=['-'],        # jpg/ofa report '-' when unavailable
        )
        df['session_id'] = session_id
        df['server']     = server
        df['datetime']   = pd.to_datetime(
            df['Date'].astype(str).str.strip() + ' ' + df['Time'].astype(str).str.strip(),
            format='%Y%m%d %H:%M:%S'
        )
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def _game_summary_row(data, session_id, filepath):
    """Flatten top-level JSON into one row (config + aggregate agent stats)."""
    cfg = data.get('config', {})
    a0  = data.get('agent_0', {})
    a1  = data.get('agent_1', {})
    return {
        'session_id':             session_id,
        'run_name':               os.path.splitext(os.path.basename(filepath))[0],
        'timestamp':              data.get('timestamp'),
        'hostname':               data.get('hostname'),
        'username':               data.get('username'),
        'host_0':                 data.get('host_0'),
        'host_1':                 data.get('host_1'),
        'elapsed_seconds':        data.get('elapsed_seconds'),
        # config
        'num_episodes':           cfg.get('num_episodes'),
        'rounds_per_episode':     cfg.get('rounds_per_episode'),
        'total_rounds':           cfg.get('total_rounds'),
        'history_window_size':    cfg.get('history_window_size'),
        'temperature':            cfg.get('temperature'),
        'reset_between_episodes': cfg.get('reset_between_episodes'),
        'reflection_type':        cfg.get('reflection_type'),
        'model_0':                cfg.get('model_0'),
        'model_1':                cfg.get('model_1'),
        # agent aggregates
        'a0_total_score':         a0.get('total_score'),
        'a0_total_cooperations':  a0.get('total_cooperations'),
        'a0_cooperation_rate':    a0.get('overall_cooperation_rate'),
        'a1_total_score':         a1.get('total_score'),
        'a1_total_cooperations':  a1.get('total_cooperations'),
        'a1_cooperation_rate':    a1.get('overall_cooperation_rate'),
    }


def _episode_rows(data, session_id, run_name):
    """One row per episode (includes per-episode scores, cooperation rates, reflections)."""
    rows = []
    for ep in data.get('episodes', []):
        a0 = ep.get('agent_0', {})
        a1 = ep.get('agent_1', {})
        rows.append({
            'session_id':          session_id,
            'run_name':            run_name,
            'episode':             ep.get('episode'),
            'a0_episode_score':    a0.get('episode_score'),
            'a0_cooperations':     a0.get('cooperations'),
            'a0_cooperation_rate': a0.get('cooperation_rate'),
            'a0_reflection':       a0.get('reflection'),
            'a1_episode_score':    a1.get('episode_score'),
            'a1_cooperations':     a1.get('cooperations'),
            'a1_cooperation_rate': a1.get('cooperation_rate'),
            'a1_reflection':       a1.get('reflection'),
        })
    return rows


def _round_rows(data, session_id, run_name):
    """One row per round (actions, reasoning, payoffs, cumulative episode scores)."""
    rows = []
    for ep in data.get('episodes', []):
        ep_num = ep.get('episode')
        for r in ep.get('rounds', []):
            rows.append({
                'session_id':      session_id,
                'run_name':        run_name,
                'episode':         ep_num,
                'round':           r.get('round'),
                'a0_action':       r.get('agent_0_action'),
                'a1_action':       r.get('agent_1_action'),
                'a0_reasoning':    r.get('agent_0_reasoning'),
                'a1_reasoning':    r.get('agent_1_reasoning'),
                'a0_payoff':       r.get('agent_0_payoff'),
                'a1_payoff':       r.get('agent_1_payoff'),
                'a0_episode_score': r.get('agent_0_episode_score'),
                'a1_episode_score': r.get('agent_1_episode_score'),
            })
    return rows


def load_games_json(games_path):
    """Parse all episodic_game_*.json files under games_path.
    Returns (summary_df, episodes_df, rounds_df) — three levels of granularity.
    Session folders are named YYYYMMDD_HHMMSS (the shared join key).
    """
    summary_rows, ep_rows, rnd_rows = [], [], []

    for session_dir in glob.glob(os.path.join(games_path, '*/')):
        session_id = os.path.basename(os.path.normpath(session_dir))
        for f in glob.glob(os.path.join(session_dir, 'episodic_game_*.json')):
            with open(f) as fh:
                data = json.load(fh)
            run_name = os.path.splitext(os.path.basename(f))[0]
            summary_rows.append(_game_summary_row(data, session_id, f))
            ep_rows.extend(_episode_rows(data, session_id, run_name))
            rnd_rows.extend(_round_rows(data, session_id, run_name))

    return pd.DataFrame(summary_rows), pd.DataFrame(ep_rows), pd.DataFrame(rnd_rows)


def load_games_log(games_path):
    """Index of all .log files. Data overlaps with JSON; useful for error-checking.
    Returns a DataFrame with session_id, run_name, log_path columns.
    """
    rows = []
    for session_dir in glob.glob(os.path.join(games_path, '*/')):
        session_id = os.path.basename(os.path.normpath(session_dir))
        for f in glob.glob(os.path.join(session_dir, 'episodic_game_*.log')):
            rows.append({
                'session_id': session_id,
                'run_name':   os.path.splitext(os.path.basename(f))[0],
                'log_path':   f,
            })
    return pd.DataFrame(rows)

In [ ]:
# ── Load all data ─────────────────────────────────────────────────────────────

testing_metrics_csv       = load_metrics_csv(testing_metrics_path)
experimenting_metrics_csv = load_metrics_csv(expirementing_metrics_path)

testing_metrics_gpu       = load_metrics_gpu(testing_metrics_path)
experimenting_metrics_gpu = load_metrics_gpu(expirementing_metrics_path)

(testing_games_summary,
 testing_games_episodes,
 testing_games_rounds)       = load_games_json(testing_games_path)

(experimenting_games_summary,
 experimenting_games_episodes,
 experimenting_games_rounds)  = load_games_json(expirementing_games_path)

testing_games_log             = load_games_log(testing_games_path)
experimenting_games_log       = load_games_log(expirementing_games_path)

# ── Row counts ────────────────────────────────────────────────────────────────
for mode, dfs in [
    ('testing',       [testing_metrics_csv, testing_metrics_gpu,
                       testing_games_summary, testing_games_episodes,
                       testing_games_rounds, testing_games_log]),
    ('experimenting', [experimenting_metrics_csv, experimenting_metrics_gpu,
                       experimenting_games_summary, experimenting_games_episodes,
                       experimenting_games_rounds, experimenting_games_log]),
]:
    names = ['metrics_csv', 'metrics_gpu', 'games_summary',
             'games_episodes', 'games_rounds', 'games_log']
    print(f"=== {mode} ===")
    for name, df in zip(names, dfs):
        print(f"  {name:<20} {len(df):>6} rows")

---
## Session Selection & Active DataFrames

**Edit the cell below** to scope all downstream analysis, then re-run from here down.

All analysis cells use `active_*` DataFrames — never the raw `testing_*` / `experimenting_*` ones directly.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EDIT THIS CELL to change which data all downstream cells operate on.
# Re-run this cell and everything below it after making changes.
# ══════════════════════════════════════════════════════════════════════════════

selected_mode = 'both'  # 'testing' | 'experimenting' | 'both'
start_date    = None    # 'YYYYMMDD' lower bound (inclusive), or None
end_date      = None    # 'YYYYMMDD' upper bound (inclusive), or None
session_ids   = None    # e.g. ['20260328_134303', '20260329_140532'], or None for all

# Run label filter — matches against the extracted label prefix of each run_name
# (i.e. the test case label from TESTS[], with _run{N} suffix stripped).
# Uses prefix matching: 'concurrency' matches 'concurrency_1', 'concurrency_5', etc.
# Leave as None to include all runs.
#
# Examples:
#   run_labels = ['concurrency_1', 'concurrency_5']   # exact label names
#   run_labels = ['concurrency']                       # all concurrency_* labels
#   run_labels = ['temp_sweep']                        # all vary= sweep runs for that label
run_labels = None

# Grouping axis for Section 1 charts.
#   'session'           — group by session_id (default; preserves original behavior)
#   'label'             — group by label prefix; merges same-label runs across sessions
#   'session_and_label' — group by session_id + label; never merges across sessions
group_by = 'session'

# Parameter filters — keys match metrics_csv column names (after resolve_defaults).
# Each value can be a single value or a list of acceptable values.
# Only games matching ALL specified filters are included. Leave as {} for all.
#
# Available keys: temp, episodes, rounds, window, model_0, model_1, host_0, host_1
#
# Examples:
#   param_filters = {'temp': 0.8}
#   param_filters = {'episodes': 50, 'window': 20}
#   param_filters = {'temp': [0.3, 0.7, 1.2]}
param_filters = {}

In [ ]:
import re

# ── helpers (not user-editable) ───────────────────────────────────────────────

def filter_df(df, id_col='session_id'):
    """Filter a DataFrame to the current date/session selection variables above."""
    if df.empty:
        return df.copy()
    mask = pd.Series(True, index=df.index)
    if start_date:
        mask &= df[id_col].str[:8] >= start_date
    if end_date:
        mask &= df[id_col].str[:8] <= end_date
    if session_ids:
        mask &= df[id_col].isin(session_ids)
    return df[mask].copy()


def extract_group_label(run_name, session_id):
    """Extract the test case label from a run_name.
    Strips the 'episodic_game_{session_id}_' prefix and '_run{N}' suffix.
    Falls back to stripping any 'episodic_game_YYYYMMDD_HHMMSS_' prefix.
    """
    prefix = f'episodic_game_{session_id}_'
    if isinstance(run_name, str) and run_name.startswith(prefix):
        label = run_name[len(prefix):]
    else:
        label = re.sub(r'^episodic_game_\d{8}_\d{6}_', '', str(run_name))
    return re.sub(r'_run\d+$', '', label)


def apply_param_filters(resolved_metrics_df):
    """Return run_name set satisfying param_filters, or None if no filter."""
    if not param_filters:
        return None
    mask = pd.Series(True, index=resolved_metrics_df.index)
    for col, val in param_filters.items():
        if col not in resolved_metrics_df.columns:
            raise KeyError(f"param_filters key '{col}' not found in metrics_csv columns.")
        if isinstance(val, list):
            mask &= resolved_metrics_df[col].isin(val)
        else:
            mask &= resolved_metrics_df[col] == val
    return set(resolved_metrics_df.loc[mask, 'run_name'])


def apply_run_label_filter(metrics_df):
    """Return run_name set whose group_label starts with any entry in run_labels.
    Returns None if run_labels is None/empty (no restriction).
    Requires group_label column to already exist on metrics_df.
    """
    if not run_labels:
        return None
    mask = metrics_df['group_label'].apply(
        lambda lbl: any(lbl.startswith(prefix) for prefix in run_labels)
    )
    return set(metrics_df.loc[mask, 'run_name'])


def _with_mode_col(df, label):
    df = df.copy()
    df.insert(0, 'mode', label)
    return df


def select_mode(testing_df, experimenting_df):
    """Return the right DataFrame(s) based on selected_mode.
    If 'both', prepends a 'mode' column so rows stay distinguishable.
    """
    if selected_mode == 'testing':
        return testing_df.copy()
    elif selected_mode == 'experimenting':
        return experimenting_df.copy()
    elif selected_mode == 'both':
        return pd.concat(
            [_with_mode_col(testing_df, 'testing'),
             _with_mode_col(experimenting_df, 'experimenting')],
            ignore_index=True
        )
    else:
        raise ValueError(f"selected_mode must be 'testing', 'experimenting', or 'both' — got {selected_mode!r}")


def filter_by_run_names(df, run_names, id_col='run_name'):
    """Restrict df to rows in the allowed run_name set.
    Returns df unchanged if run_names is None (no filter active).
    """
    if run_names is None:
        return df
    if id_col in df.columns:
        return df[df[id_col].isin(run_names)].copy()
    return df

In [ ]:
# ── Resolve defaults + produce active_* DataFrames ───────────────────────────

_DEFAULT_RESOLUTION = {
    'episodes': 'num_episodes',
    'rounds':   'rounds_per_episode',
    'window':   'history_window_size',
    'temp':     'temperature',
    'model_0':  'model_0',
    'model_1':  'model_1',
    'host_0':   'host_0',
    'host_1':   'host_1',
}

def resolve_defaults(metrics_df, summary_df):
    """Replace 'default' strings in metrics_csv config columns with JSON values."""
    if metrics_df.empty:
        return metrics_df.copy()
    needed = list(set(_DEFAULT_RESOLUTION.values()))
    summary_sub = summary_df[['run_name'] + needed].rename(
        columns={c: f'_json_{c}' for c in needed}
    )
    df = metrics_df.merge(summary_sub, on='run_name', how='left')
    for metrics_col, summary_col in _DEFAULT_RESOLUTION.items():
        if metrics_col not in df.columns:
            continue
        df[metrics_col] = df[metrics_col].astype(object)
        mask = df[metrics_col].astype(str).str.strip().str.lower() == 'default'
        df.loc[mask, metrics_col] = df.loc[mask, f'_json_{summary_col}']
    df = df.drop(columns=[c for c in df.columns if c.startswith('_json_')])
    for col in ['episodes', 'rounds', 'window']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    df['temp'] = pd.to_numeric(df['temp'], errors='coerce')
    return df


# ── 1. Mode selection ─────────────────────────────────────────────────────────
_raw_metrics_csv    = select_mode(testing_metrics_csv,    experimenting_metrics_csv)
_raw_metrics_gpu    = select_mode(testing_metrics_gpu,    experimenting_metrics_gpu)
_raw_games_summary  = select_mode(testing_games_summary,  experimenting_games_summary)
_raw_games_episodes = select_mode(testing_games_episodes, experimenting_games_episodes)
_raw_games_rounds   = select_mode(testing_games_rounds,   experimenting_games_rounds)
_raw_games_log      = select_mode(testing_games_log,      experimenting_games_log)

# ── 2. Resolve 'default' config strings ──────────────────────────────────────
_full_summary_for_lookup = pd.concat(
    [testing_games_summary, experimenting_games_summary], ignore_index=True
)
_resolved_metrics_csv = resolve_defaults(_raw_metrics_csv, _full_summary_for_lookup)

# ── 3. Add group_label column (used by run_labels filter and Section 1 charts) ─
_resolved_metrics_csv['group_label'] = _resolved_metrics_csv.apply(
    lambda r: extract_group_label(r['run_name'], r['session_id']), axis=1
)

# ── 4. Date / session_id filter ───────────────────────────────────────────────
_filtered_metrics_csv    = filter_df(_resolved_metrics_csv)
_filtered_metrics_gpu    = filter_df(_raw_metrics_gpu)
_filtered_games_summary  = filter_df(_raw_games_summary)
_filtered_games_episodes = filter_df(_raw_games_episodes)
_filtered_games_rounds   = filter_df(_raw_games_rounds)
_filtered_games_log      = filter_df(_raw_games_log)

# ── 5. Run label filter ───────────────────────────────────────────────────────
_matched_labels = apply_run_label_filter(_filtered_metrics_csv)
_filtered_metrics_csv = filter_by_run_names(_filtered_metrics_csv, _matched_labels)

# ── 6. Param filter ───────────────────────────────────────────────────────────
_matched_runs = apply_param_filters(_filtered_metrics_csv)

active_metrics_csv    = filter_by_run_names(_filtered_metrics_csv, _matched_runs)
active_metrics_gpu    = (
    filter_by_run_names(_filtered_metrics_gpu, _matched_runs, id_col='session_id')
    if _matched_runs is not None else _filtered_metrics_gpu
)
active_games_summary  = filter_by_run_names(_filtered_games_summary,  _matched_runs)
active_games_episodes = filter_by_run_names(_filtered_games_episodes, _matched_runs)
active_games_rounds   = filter_by_run_names(_filtered_games_rounds,   _matched_runs)
active_games_log      = filter_by_run_names(_filtered_games_log,      _matched_runs)

# ── 7. Derive _group_col for Section 1 charts ─────────────────────────────────
if group_by == 'session':
    _group_col = 'session_id'
elif group_by == 'label':
    _group_col = 'group_label'
elif group_by == 'session_and_label':
    active_metrics_csv['session_label'] = (
        active_metrics_csv['session_id'] + ' / ' + active_metrics_csv['group_label']
    )
    _group_col = 'session_label'
else:
    raise ValueError(f"group_by must be 'session', 'label', or 'session_and_label' — got {group_by!r}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Selection:  mode={selected_mode!r}  start={start_date}  end={end_date}  session_ids={session_ids}")
print(f"            run_labels={run_labels}  group_by={group_by!r}")
print(f"            param_filters={param_filters}")
print(f"            _group_col={_group_col!r}")
print()
for name, df in [
    ('active_metrics_csv',    active_metrics_csv),
    ('active_metrics_gpu',    active_metrics_gpu),
    ('active_games_summary',  active_games_summary),
    ('active_games_episodes', active_games_episodes),
    ('active_games_rounds',   active_games_rounds),
    ('active_games_log',      active_games_log),
]:
    sessions = df['session_id'].nunique() if 'session_id' in df.columns and not df.empty else '—'
    runs     = df['run_name'].nunique()   if 'run_name'   in df.columns and not df.empty else '—'
    print(f"  {name:<25} {len(df):>6} rows   {sessions} session(s)   {runs} run(s)")

In [ ]:
# ── Quick previews ────────────────────────────────────────────────────────────
# print("--- testing_metrics_csv ---")
# display(testing_metrics_csv.head(3))

# print("--- testing_metrics_gpu ---")
# display(testing_metrics_gpu.head(3))

# print("--- testing_games_summary ---")
# display(testing_games_summary.head(3))

# print("--- testing_games_episodes ---")
# display(testing_games_episodes.head(3))

# print("--- testing_games_rounds ---")
# display(testing_games_rounds.head(3))
print("--- metrics csv ---")
display(active_metrics_csv.head())

print("--- metrics gpu ---")
display(active_metrics_gpu.head())

print("--- games summary ---")
display(active_games_summary.head())

print("--- games episodes ---")
display(active_games_episodes.head())

print("--- games rounds ---")
display(active_games_rounds.head())

print("--- games log ---")
display(active_games_log.head())

---
## Section 1: Runtime & Parallelism

Answers: how does wall-clock time scale with concurrent game count, and is the GPU a bottleneck?

All cells use `active_metrics_csv` and `active_metrics_gpu` — re-run the selection cell to change scope.

In [ ]:
import numpy as np

# ── 1a. Batch-level timing summary ────────────────────────────────────────────
# Groups by _group_col (set in the resolve cell by group_by variable).
if active_metrics_csv.empty:
    print('No data in active selection.')
else:
    timing = (
        active_metrics_csv
        .groupby(_group_col)
        .agg(
            batch_start=('start_time', 'min'),
            batch_end=('end_time', 'max'),
            n_runs=('run_name', 'count'),
            total_cpu_s=('elapsed_seconds', 'sum'),
            avg_game_s=('elapsed_seconds', 'mean'),
            min_game_s=('elapsed_seconds', 'min'),
            max_game_s=('elapsed_seconds', 'max'),
            failed=('exit_code', lambda x: (x != 0).sum()),
        )
        .reset_index()
    )
    timing['wall_time_s']       = timing['batch_end'] - timing['batch_start']
    timing['wall_time_min']     = (timing['wall_time_s'] / 60).round(2)
    timing['avg_game_min']      = (timing['avg_game_s'] / 60).round(2)
    timing['parallelism_ratio'] = (timing['total_cpu_s'] / timing['wall_time_s']).round(2)

    display(
        timing[[_group_col, 'n_runs', 'wall_time_min', 'avg_game_min',
                 'min_game_s', 'max_game_s', 'parallelism_ratio', 'failed']]
        .sort_values(_group_col)
        .reset_index(drop=True)
    )

    failed_total = timing['failed'].sum()
    if failed_total > 0:
        print(f'\n⚠  {int(failed_total)} failed run(s) detected (exit_code != 0). Check active_games_log for paths.')

In [ ]:
# ── 1b. Gantt chart — when did each game run? ────────────────────────────────
# One subplot per group (defined by _group_col / group_by).
if active_metrics_csv.empty:
    print('No data in active selection.')
else:
    groups = sorted(active_metrics_csv[_group_col].unique())
    n_total_runs = len(active_metrics_csv)
    fig, axes = plt.subplots(
        len(groups), 1,
        figsize=(14, max(4, n_total_runs * 0.35 + len(groups))),
        squeeze=False
    )

    for ax, grp in zip(axes.flat, groups):
        grp_df = (active_metrics_csv[active_metrics_csv[_group_col] == grp]
                  .copy().sort_values('start_time').reset_index(drop=True))
        batch_start = grp_df['start_time'].min()

        for i, row in grp_df.iterrows():
            rel_start = row['start_time'] - batch_start
            ax.barh(i, row['elapsed_seconds'], left=rel_start,
                    color='steelblue', alpha=0.75, edgecolor='white', linewidth=0.5)
            ax.text(rel_start + row['elapsed_seconds'] + 2, i,
                    f"{row['elapsed_seconds']:.0f}s", va='center', fontsize=6)

        # Short label: strip episodic_game_YYYYMMDD_HHMMSS_ prefix
        import re as _re
        short = [_re.sub(r'^episodic_game_\d{8}_\d{6}_', '', r['run_name'])
                 for _, r in grp_df.iterrows()]
        ax.set_yticks(range(len(grp_df)))
        ax.set_yticklabels(short, fontsize=7)
        ax.set_xlabel('Seconds since group start')
        ax.set_title(f'{_group_col}: {grp}  —  {len(grp_df)} runs', fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Gantt Chart: Game Execution Windows', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 1c. Concurrency over time + GPU SM utilization overlay ──────────────────
def _concurrency_events(grp_df):
    """Build a step-function of concurrent game count over time for one group."""
    batch_start = grp_df['start_time'].min()
    events = []
    for _, row in grp_df.iterrows():
        events.append((row['start_time'] - batch_start, +1))
        events.append((row['end_time']   - batch_start, -1))
    ev = (pd.DataFrame(events, columns=['t', 'delta'])
          .sort_values('t').reset_index(drop=True))
    ev['concurrency'] = ev['delta'].cumsum()
    return ev

if active_metrics_csv.empty:
    print('No data in active selection.')
else:
    groups = sorted(active_metrics_csv[_group_col].unique())
    fig, axes = plt.subplots(len(groups), 1,
                             figsize=(14, max(4, len(groups) * 3.5)),
                             squeeze=False)

    for ax, grp in zip(axes.flat, groups):
        grp_df      = active_metrics_csv[active_metrics_csv[_group_col] == grp]
        batch_start = grp_df['start_time'].min()
        ev          = _concurrency_events(grp_df)

        ax.step(ev['t'], ev['concurrency'], where='post', color='steelblue', linewidth=2)
        ax.fill_between(ev['t'], ev['concurrency'], step='post', color='steelblue', alpha=0.15)
        ax.set_ylabel('Concurrent games', color='steelblue')
        ax.set_ylim(bottom=0)
        ax.set_xlabel('Seconds since group start')
        ax.set_title(f'{_group_col}: {grp}', fontweight='bold')
        ax.grid(alpha=0.3)

        # GPU overlay — collect all session_ids that belong to this group
        grp_sessions = grp_df['session_id'].unique()
        gpu_grp = active_metrics_gpu[active_metrics_gpu['session_id'].isin(grp_sessions)]
        if not gpu_grp.empty:
            batch_start_dt = pd.Timestamp(batch_start, unit='s')
            gpu_grp = gpu_grp.copy()
            gpu_grp['t_rel'] = (gpu_grp['datetime'] - batch_start_dt).dt.total_seconds()
            ax2 = ax.twinx()
            for gidx in sorted(gpu_grp['gpu_idx'].unique()):
                g = gpu_grp[gpu_grp['gpu_idx'] == gidx]
                ax2.plot(g['t_rel'], g['sm_pct'], alpha=0.6, linewidth=1,
                         label=f'GPU {gidx} SM%')
            ax2.set_ylabel('GPU SM utilization (%)', color='darkorange')
            ax2.set_ylim(0, 105)
            ax2.tick_params(axis='y', labelcolor='darkorange')
            ax2.legend(loc='upper right', fontsize=7)

    plt.suptitle('Concurrency Over Time (+ GPU SM Utilization)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 1d. Wall time scaling + per-group runtime distribution ──────────────────
if active_metrics_csv.empty:
    print('No data in active selection.')
else:
    groups = sorted(active_metrics_csv[_group_col].unique())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: wall_time vs n_runs with reference lines
    ax = axes[0]
    single_ref = timing['avg_game_s'].mean()
    x_max = timing['n_runs'].max() * 1.15
    x_ref = np.linspace(0, x_max, 200)
    ax.plot(x_ref, np.full_like(x_ref, single_ref),
            'g--', alpha=0.6, label=f'Ideal parallel ({single_ref:.0f}s)')
    ax.plot(x_ref, x_ref * single_ref,
            'r--', alpha=0.6, label='Ideal serial')
    ax.scatter(timing['n_runs'], timing['wall_time_s'], s=80, zorder=3, color='steelblue')
    for _, row in timing.iterrows():
        ax.annotate(str(row[_group_col]),
                    (row['n_runs'], row['wall_time_s']),
                    textcoords='offset points', xytext=(6, 4), fontsize=7)
    ax.set_xlabel('Number of games in group')
    ax.set_ylabel('Wall time (seconds)')
    ax.set_title(f'Parallelism Scaling  (grouped by {_group_col})')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Right: per-game elapsed time distribution per group (boxplot)
    ax = axes[1]
    data_by_grp = [active_metrics_csv[active_metrics_csv[_group_col] == g]['elapsed_seconds'].values
                   for g in groups]
    bp = ax.boxplot(data_by_grp, labels=groups, patch_artist=True, notch=False)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    ax.set_xlabel(_group_col)
    ax.set_ylabel('Elapsed seconds per game')
    ax.set_title('Per-Game Runtime Distribution')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Runtime Scaling & Distribution', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Section 2: Parameter Sensitivity

Answers: which config parameters affect runtime or cooperation behavior, and by how much?

Populated by `vary=` sweep runs. If no parameter variation exists in the active selection, cells will say so.

In [ ]:
# ── 2a. Runtime vs. varied parameter ─────────────────────────────────────────
CONFIG_COLS = ['temp', 'episodes', 'rounds', 'window']
varied = [c for c in CONFIG_COLS
          if c in active_metrics_csv.columns and active_metrics_csv[c].nunique() > 1]

if not varied:
    print("No parameter variation detected in active selection.")
    print("Run a vary= sweep test then re-scope the selection to populate this section.")
else:
    print(f"Varied parameters: {varied}\n")
    for param in varied:
        grp = (active_metrics_csv.groupby(param)['elapsed_seconds']
               .agg(mean='mean', std='std', n='count').reset_index())

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        # Mean ± std
        axes[0].errorbar(grp[param], grp['mean'], yerr=grp['std'],
                         fmt='o-', capsize=4, color='steelblue')
        axes[0].set_xlabel(param)
        axes[0].set_ylabel('Elapsed seconds')
        axes[0].set_title(f'Mean runtime ± 1 std  ({param})')
        axes[0].grid(alpha=0.3)

        # All individual runs
        axes[1].scatter(active_metrics_csv[param], active_metrics_csv['elapsed_seconds'],
                        alpha=0.45, color='steelblue', edgecolors='white', linewidths=0.4)
        axes[1].set_xlabel(param)
        axes[1].set_ylabel('Elapsed seconds')
        axes[1].set_title(f'All runs  ({param})')
        axes[1].grid(alpha=0.3)

        plt.suptitle(f'Runtime Sensitivity: {param}', fontsize=12)
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 2b. Cooperation rate + score efficiency vs. varied parameter ───────────────
if not varied:
    print("No parameter variation detected — skipping behavioral sensitivity plots.")
else:
    # Join resolved config params onto games_summary via run_name
    summary_with_params = active_games_summary.merge(
        active_metrics_csv[['run_name'] + varied],
        on='run_name', how='inner'
    )
    summary_with_params['score_efficiency'] = (
        (summary_with_params['a0_total_score'] + summary_with_params['a1_total_score'])
        / (summary_with_params['total_rounds'] * 6)
    )

    for param in varied:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        # Cooperation rate per agent
        ax = axes[0]
        for agent, col, color in [('Agent 0', 'a0_cooperation_rate', 'steelblue'),
                                   ('Agent 1', 'a1_cooperation_rate', 'coral')]:
            ax.scatter(summary_with_params[param], summary_with_params[col],
                       alpha=0.45, color=color, label=agent, s=40,
                       edgecolors='white', linewidths=0.4)
            means = summary_with_params.groupby(param)[col].mean()
            ax.plot(means.index, means.values, color=color, linewidth=1.5)

        ax.set_xlabel(param)
        ax.set_ylabel('Overall cooperation rate')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f'Cooperation Rate vs. {param}')
        ax.legend()
        ax.grid(alpha=0.3)

        # Score efficiency
        ax = axes[1]
        ax.scatter(summary_with_params[param], summary_with_params['score_efficiency'],
                   alpha=0.45, color='mediumseagreen', s=40,
                   edgecolors='white', linewidths=0.4)
        means_eff = summary_with_params.groupby(param)['score_efficiency'].mean()
        ax.plot(means_eff.index, means_eff.values, color='mediumseagreen', linewidth=1.5)
        ax.set_xlabel(param)
        ax.set_ylabel('Score efficiency  (1 = all mutual cooperation)')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f'Score Efficiency vs. {param}')
        ax.grid(alpha=0.3)

        plt.suptitle(f'Behavioral Sensitivity: {param}', fontsize=12)
        plt.tight_layout()
        plt.show()

---
## Section 3: Stagger Analysis

Answers: does staggering game launches reduce per-game slowdowns, and does it improve or worsen total wall time?

**Configure `stagger_sessions` in the cell below** to map each session_id to a human-readable stagger label before running.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EDIT: map session_id → stagger label for the sessions you want to compare.
# Leave as {} if stagger tests haven't been run yet.
#
# Example (after running Group 2 tests from the suggested battery):
#   stagger_sessions = {
#       '20260401_100000': 'no delay',
#       '20260401_110000': '15s delay',
#       '20260401_120000': 'wait',
#   }
# ══════════════════════════════════════════════════════════════════════════════
stagger_sessions = {}

# ── 3a. Per-game elapsed time by stagger group ────────────────────────────────
if not stagger_sessions:
    print("stagger_sessions is empty — configure the dict above once stagger tests are available.")
else:
    stagger_df = active_metrics_csv[
        active_metrics_csv['session_id'].isin(stagger_sessions)
    ].copy()
    stagger_df['stagger_label'] = stagger_df['session_id'].map(stagger_sessions)

    labels  = list(dict.fromkeys(stagger_sessions.values()))   # preserve insertion order
    data    = [stagger_df[stagger_df['stagger_label'] == lbl]['elapsed_seconds'].values
               for lbl in labels]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Boxplot
    bp = axes[0].boxplot(data, labels=labels, patch_artist=True)
    colors = plt.cm.Set2(range(len(labels)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0].set_xlabel('Stagger condition')
    axes[0].set_ylabel('Elapsed seconds per game')
    axes[0].set_title('Per-Game Runtime by Stagger Condition')
    axes[0].grid(axis='y', alpha=0.3)

    # Wall time comparison
    wall_by_stagger = (
        stagger_df.groupby(['stagger_label', 'session_id'])
        .agg(batch_start=('start_time', 'min'), batch_end=('end_time', 'max'))
        .reset_index()
    )
    wall_by_stagger['wall_time_s'] = wall_by_stagger['batch_end'] - wall_by_stagger['batch_start']
    wall_by_label = wall_by_stagger.groupby('stagger_label')['wall_time_s'].mean().reindex(labels)

    axes[1].bar(labels, wall_by_label.values,
                color=[plt.cm.Set2(i) for i in range(len(labels))], alpha=0.75, edgecolor='white')
    axes[1].set_xlabel('Stagger condition')
    axes[1].set_ylabel('Batch wall time (seconds)')
    axes[1].set_title('Total Wall Time by Stagger Condition')
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Stagger Comparison', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 3b. GPU memory (fb_MB) over time per stagger group ───────────────────────
if not stagger_sessions:
    print("stagger_sessions is empty — configure the dict in the cell above.")
else:
    gpu_stagger = active_metrics_gpu[
        active_metrics_gpu['session_id'].isin(stagger_sessions)
    ].copy()

    if gpu_stagger.empty:
        print("No GPU telemetry found for the specified stagger sessions.")
    else:
        gpu_stagger['stagger_label'] = gpu_stagger['session_id'].map(stagger_sessions)
        labels = list(dict.fromkeys(stagger_sessions.values()))
        colors = plt.cm.Set2(range(len(labels)))

        gpu_idxs = sorted(gpu_stagger['gpu_idx'].unique())
        fig, axes = plt.subplots(len(gpu_idxs), 1,
                                 figsize=(14, max(4, len(gpu_idxs) * 3.5)),
                                 squeeze=False)

        for ax, gidx in zip(axes.flat, gpu_idxs):
            g_data = gpu_stagger[gpu_stagger['gpu_idx'] == gidx]
            for color, lbl in zip(colors, labels):
                grp = g_data[g_data['stagger_label'] == lbl].copy()
                if grp.empty:
                    continue
                # Align each session to t=0 using its session's batch start
                sid = grp['session_id'].iloc[0]
                sess_start = (active_metrics_csv[active_metrics_csv['session_id'] == sid]
                              ['start_time'].min())
                sess_start_dt = pd.Timestamp(sess_start, unit='s')
                grp['t_rel'] = (grp['datetime'] - sess_start_dt).dt.total_seconds()
                ax.plot(grp['t_rel'], grp['fb_MB'], color=color, alpha=0.75,
                        linewidth=1.2, label=lbl)

            ax.set_xlabel('Seconds since batch start')
            ax.set_ylabel('GPU frame-buffer memory (MB)')
            ax.set_title(f'GPU {gidx} — Memory Usage by Stagger Condition')
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)

        plt.suptitle('GPU Memory Trace by Stagger Condition', fontsize=13, y=1.01)
        plt.tight_layout()
        plt.show()

---
## Section 4: Behavioral Analysis

Answers: do agents learn to cooperate over time, what outcomes dominate, and is there a structural advantage for one agent?

Uses `active_games_episodes` and `active_games_rounds`.

In [ ]:
# ── 4a. Episode learning curves ───────────────────────────────────────────────
if active_games_episodes.empty:
    print("No episode data in active selection.")
else:
    ep_stats = (
        active_games_episodes
        .groupby('episode')[['a0_cooperation_rate', 'a1_cooperation_rate']]
        .agg(['mean', 'std'])
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(11, 5))
    for agent, col, color in [('Agent 0', 'a0_cooperation_rate', 'steelblue'),
                               ('Agent 1', 'a1_cooperation_rate', 'coral')]:
        mean = ep_stats[col]['mean']
        std  = ep_stats[col]['std'].fillna(0)
        ax.plot(ep_stats['episode'], mean, label=agent, color=color, linewidth=2)
        ax.fill_between(ep_stats['episode'], mean - std, mean + std,
                        alpha=0.15, color=color)

    ax.set_xlabel('Episode number')
    ax.set_ylabel('Cooperation rate')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('Episode Learning Curves  (mean ± 1 std across all active runs)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Numeric summary: first vs. last episode
    first = active_games_episodes[active_games_episodes['episode'] == active_games_episodes['episode'].min()]
    last  = active_games_episodes[active_games_episodes['episode'] == active_games_episodes['episode'].max()]
    print(f"Episode {first['episode'].iloc[0]}  →  "
          f"A0 coop {first['a0_cooperation_rate'].mean():.2f}, "
          f"A1 coop {first['a1_cooperation_rate'].mean():.2f}")
    print(f"Episode {last['episode'].iloc[0]}   →  "
          f"A0 coop {last['a0_cooperation_rate'].mean():.2f}, "
          f"A1 coop {last['a1_cooperation_rate'].mean():.2f}")

In [ ]:
# ── 4b. Round outcome distribution (CC / CD / DC / DD) ───────────────────────
if active_games_rounds.empty:
    print("No round data in active selection.")
else:
    def _outcome(row):
        a0, a1 = row['a0_action'], row['a1_action']
        if a0 == 'COOPERATE' and a1 == 'COOPERATE': return 'CC'
        if a0 == 'COOPERATE' and a1 == 'DEFECT':    return 'CD'
        if a0 == 'DEFECT'    and a1 == 'COOPERATE': return 'DC'
        return 'DD'

    rounds = active_games_rounds.copy()
    rounds['outcome'] = rounds.apply(_outcome, axis=1)

    outcome_dist = (
        rounds.groupby('session_id')['outcome']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=['CC', 'CD', 'DC', 'DD'], fill_value=0)
    )

    OUTCOME_COLORS = {'CC': '#2ecc71', 'CD': '#e67e22', 'DC': '#3498db', 'DD': '#e74c3c'}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Stacked bar per session
    outcome_dist.plot(kind='bar', stacked=True, ax=axes[0],
                      color=[OUTCOME_COLORS[c] for c in outcome_dist.columns],
                      alpha=0.85, edgecolor='white')
    axes[0].set_xlabel('Session')
    axes[0].set_ylabel('Fraction of rounds')
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Outcome Distribution by Session')
    axes[0].tick_params(axis='x', rotation=25)
    axes[0].legend(title='Outcome', bbox_to_anchor=(1.01, 1), loc='upper left')
    axes[0].grid(axis='y', alpha=0.3)

    # Overall pie across all active rounds
    overall = rounds['outcome'].value_counts()
    wedge_order = [o for o in ['CC', 'CD', 'DC', 'DD'] if o in overall.index]
    axes[1].pie(overall[wedge_order],
                labels=wedge_order,
                colors=[OUTCOME_COLORS[o] for o in wedge_order],
                autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Overall Outcome Mix (all active rounds)')

    plt.suptitle('Round Outcome Distribution', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 4c. Agent asymmetry (score diff + cooperation rate diff) ──────────────────
if active_games_summary.empty:
    print("No summary data in active selection.")
else:
    summary = active_games_summary.copy()
    summary['score_diff'] = summary['a0_total_score'] - summary['a1_total_score']
    summary['coop_diff']  = summary['a0_cooperation_rate'] - summary['a1_cooperation_rate']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, col, label, color in [
        (axes[0], 'score_diff',
         'Agent 0 total score − Agent 1 total score', 'steelblue'),
        (axes[1], 'coop_diff',
         'Agent 0 coop rate − Agent 1 coop rate',     'coral'),
    ]:
        ax.hist(summary[col], bins=25, color=color, alpha=0.7, edgecolor='white')
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1.2)
        ax.axvline(summary[col].mean(), color='red', linestyle='-', linewidth=1.5,
                   label=f'mean = {summary[col].mean():.2f}')
        ax.set_xlabel(label)
        ax.set_ylabel('Number of games')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    axes[0].set_title('Score Asymmetry')
    axes[1].set_title('Cooperation Rate Asymmetry')

    plt.suptitle('Agent Asymmetry  (0 = perfectly symmetric)', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"Score diff   — mean: {summary['score_diff'].mean():.2f}, "
          f"std: {summary['score_diff'].std():.2f}, "
          f"% where A0 > A1: {(summary['score_diff'] > 0).mean():.1%}")
    print(f"Coop rate diff — mean: {summary['coop_diff'].mean():.3f}, "
          f"std: {summary['coop_diff'].std():.3f}")